# E041 — Histogram Equalization for Image Preprocessing

**Motivation:** Session lighting variation is a major challenge (E007 brightness aug helped). Histogram equalization (HE) or CLAHE could normalize illumination before PCA, potentially improving robustness.

**Hypothesis:** Applying CLAHE (Contrast-Limited Adaptive Histogram Equalization) before PCA will improve session generalization by normalizing lighting conditions.

**Approach:**
1. Apply CLAHE to images before feature extraction
2. Compare to standard preprocessing (E033)
3. Test CLAHE alone and CLAHE + E033 augmentations

**Configs:**
- `raw`: E033 baseline (no HE)
- `HE`: Global histogram equalization
- `CLAHE`: Adaptive CLAHE (clip=2.0, tile=8×8)
- `CLAHE+aug`: CLAHE + E033 augmentations

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
from PIL import Image, ImageOps
from scipy.ndimage import rotate
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import cv2

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
N_PCA = 50

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E033_REF = {'mean_eer': 0.51, 'std_eer': 0.36}

222 samples


In [2]:
def _find_png(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".png")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _load_image(path):
    img = Image.open(path).convert("RGB")
    if img.size != (80, 80):
        img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2)

def apply_he(x):
    """Global histogram equalization."""
    x_uint8 = np.clip(x, 0, 255).astype(np.uint8)
    x_he = ImageOps.equalize(Image.fromarray(x_uint8))
    return np.array(x_he, dtype=np.float32)

def apply_clahe(x, clip=2.0, tile=(8, 8)):
    """CLAHE - Contrast Limited Adaptive Histogram Equalization."""
    x_uint8 = np.clip(x, 0, 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile)
    x_clahe = clahe.apply(x_uint8)
    return x_clahe.astype(np.float32)

def train_image_model(df, data_dir, preprocess_fn, seed, with_aug=True):
    """Train image model with specified preprocessing."""
    rng = np.random.default_rng(seed)
    
    X, y = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row["stem"], data_dir))
        orig = preprocess_fn(orig)
        
        vecs = [orig, orig[:, ::-1].copy()]
        if with_aug:
            vecs += [np.clip(orig * rng.uniform(0.7, 1.3), 0, 255)]
            vecs += [np.clip(orig + rng.normal(0, 15, orig.shape), 0, 255)]
            # Adversarial rotation for target
            if row["label"] == 1:
                for angle in [-10, -5, 5, 10]:
                    vecs.append(rotate(orig, angle, reshape=False, order=1, mode='constant', cval=0))
        
        for v in vecs:
            X.append(v.flatten()); y.append(row["label"])
    
    X = np.array(X)
    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    X_pca = pca.fit_transform(scaler.fit_transform(X))
    clf.fit(X_pca, y)
    
    return scaler, pca, clf, preprocess_fn

def score_image(png_path, scaler, pca, clf, preprocess_fn):
    x = _load_image(png_path)
    x = preprocess_fn(x)
    x_flat = x.flatten().reshape(1, -1)
    x_pca = pca.transform(scaler.transform(x_flat))
    return float(clf.decision_function(x_pca)[0])

print('Preprocessing functions defined')

Preprocessing functions defined


## 2. Cross-validation

In [3]:
configs = {
    'raw (E033)': (lambda x: x, True),
    'HE': (apply_he, False),
    'CLAHE': (lambda x: apply_clahe(x), False),
    'CLAHE+aug': (lambda x: apply_clahe(x), True),
}

results = {}

for name, (preprocess_fn, with_aug) in configs.items():
    print(f"\n=== {name} ===")
    fold_eers = []
    
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        seed_f = SEED + fold_id
        train_df = manifest.loc[train_idx]
        
        scaler, pca, clf, _ = train_image_model(train_df, DATA, preprocess_fn, seed_f, with_aug)
        
        scores = []
        for _, row in manifest.loc[val_idx].iterrows():
            score = score_image(_find_png(row["stem"], DATA), scaler, pca, clf, preprocess_fn)
            scores.append(score)
        
        scores = np.array(scores)
        eer, _ = compute_eer(scores[y_all[val_idx] == 1], scores[y_all[val_idx] == 0])
        fold_eers.append(eer * 100)
        print(f"  Fold {fold_id}: EER={eer*100:.2f}%")
    
    results[name] = {
        'fold_eers': fold_eers,
        'mean': np.mean(fold_eers),
        'std': np.std(fold_eers),
    }
    print(f"  Mean ± Std: {np.mean(fold_eers):.2f} ± {np.std(fold_eers):.2f}%")

print("\n=== Summary ===")
for name, res in results.items():
    print(f"{name:15s}: {res['mean']:5.2f} ± {res['std']:5.2f}%")


=== raw (E033) ===


  Fold 0: EER=2.08%


  Fold 1: EER=0.83%


  Fold 2: EER=0.00%
  Mean ± Std: 0.97 ± 0.86%

=== HE ===


  Fold 0: EER=0.69%


  Fold 1: EER=8.33%
  Fold 2: EER=0.00%
  Mean ± Std: 3.01 ± 3.78%

=== CLAHE ===


  Fold 0: EER=1.39%


  Fold 1: EER=7.50%
  Fold 2: EER=0.00%
  Mean ± Std: 2.96 ± 3.26%

=== CLAHE+aug ===


  Fold 0: EER=8.47%
  Fold 1: EER=8.33%


  Fold 2: EER=0.00%
  Mean ± Std: 5.60 ± 3.96%

=== Summary ===
raw (E033)     :  0.97 ±  0.86%
HE             :  3.01 ±  3.78%
CLAHE          :  2.96 ±  3.26%
CLAHE+aug      :  5.60 ±  3.96%


## 3. Conclusion

Histogram equalization effect: [TBD]
Decision: [Adopt/Reject]